# Interval audit v2: the four remaining items

v1 measured the law. This closes the four things that stop it being publishable.

| # | Item | Gap it fills |
|---|---|---|
| 1 | Calibrate a screen tolerance | 64 of 128 flagged rows were numerical, and every construction pays per flagged row. Worth ~1,088 bits. |
| 2 | Estimate the head profile instead of capping | `fraction_exact` was 0.0, so the converse rested on an unverified upper bound. |
| 3 | Random-coordinate control | 91.3% was measured on the top 0.025% of the head by salience. |
| 4 | Larger pool, longer sweep, log `feasible_columns` | `N=256` from 384 texts is near the pool ceiling; the `-1` slope extrapolates to a single survivor at `N ~ 8,400`, untestable at 384. |

Everything routes through `interval_audit.py`, which is unit-tested against exhaustive brute
force (outer bound, exact verification, chunk invariance, estimator coverage).

**Runtime.** Feature extraction for the enlarged pool dominates: budget 30-60 min, then roughly
an hour for the measurements. Nothing here trains, adds a model, sweeps output precision, or
implements a new decoder.

**Information hygiene.** Cells marked `EVALUATOR ONLY` build the tamper and score results. The
measurement path sees only the damaged head, the retained features, the retained codes, and
public settings.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import gc, json, math, time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

import interval_audit as ia          # keep interval_audit.py beside this notebook


@dataclass(frozen=True)
class Config:
    model_id: str = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    hf_cache_dir: str = "/scratch/bbjr/skarmakar/huggingface"
    model_revision: str | None = None
    dataset_revision: str | None = None
    cache_dir: str = "repair_runs/audit_v2"
    out_dir: str = "repair_runs/audit_v2"
    pilot_seed: int = 17
    tamper_trial_id: int = 1000
    sparsity: int = 64
    output_bits: int = 8
    fixed_logit_range: float = 64.0
    rho_multiple: float = 10.0
    tamper_fraction_of_bound: float = 0.9
    constraint_slack: float = 1e-4          # interval outer-bound tolerance
    context_tokens: int = 256
    feature_batch: int = 24
    head_batch: int = 32
    cal_contexts: int = 96
    # Item 4: enlarge the pool. Requests are trimmed to what the split actually provides.
    pool_request: int = 8192
    n_grid_request: tuple = (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
    sample_with_replacement: bool = False   # v1 used replacement; distinct inputs are cleaner
    # Item 1: screen tolerances to calibrate over.
    eta_grid: tuple = (0.0, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3)
    # Item 2: estimate rather than cap.
    verify_cap: int = 200_000
    estimate_draws: int = 2048
    head_profile_rows: int = 512
    # Item 3: tamper coordinate pools.
    pool_modes: tuple = ("salient", "uniform", "salient_rows_uniform_cols",
                         "uniform_rows_salient_cols")
    sweep_modes: tuple = ("salient", "uniform")
    interval_chunk: int = 1024

CFG = Config()
assert torch.cuda.is_available(), "A CUDA GPU is required."
DEVICE = torch.device("cuda")
torch.manual_seed(CFG.pilot_seed); np.random.seed(CFG.pilot_seed)
OUT = Path(CFG.out_dir); OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path(CFG.cache_dir); CACHE.mkdir(parents=True, exist_ok=True)
R = float(CFG.fixed_logit_range)
WIDTH = 2 * R / (2 ** CFG.output_bits)
VALUES, LABELS = ia.bf16_table()
REPORT = {}
print({"out": str(OUT), "cell_width": WIDTH, "finite_bf16_values": len(VALUES)})

## Inputs

Held-out text only: the pool is drawn from validation and test, disjoint from the calibration
slice used to define the salience pools. The request is trimmed to what the splits actually
provide after the length filter, and the N grid is trimmed with it.

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", revision=CFG.dataset_revision)

def usable(split):
    return [x["text"].strip() for x in ds[split] if len(x["text"].strip()) >= 80]

held_out = list(dict.fromkeys(usable("validation") + usable("test")))   # dedup, keep order
rng = np.random.default_rng(CFG.pilot_seed + 2)
pool_n = min(CFG.pool_request, len(held_out))
pool_texts = [held_out[i] for i in rng.choice(len(held_out), pool_n, replace=False)]
cal_texts = [usable("train")[i] for i in
             np.random.default_rng(CFG.pilot_seed).choice(len(usable("train")),
                                                          CFG.cal_contexts, replace=False)]
N_GRID = tuple(n for n in CFG.n_grid_request
               if n <= (pool_n if not CFG.sample_with_replacement else CFG.pool_request))
N_MAX = max(N_GRID)
REPORT["pool"] = {"requested": CFG.pool_request, "available_held_out": len(held_out),
                  "pool_used": pool_n, "n_grid": list(N_GRID), "N_max": N_MAX,
                  "with_replacement": CFG.sample_with_replacement}
print(json.dumps(REPORT["pool"], indent=2))
if pool_n < CFG.pool_request:
    print(f"\nNOTE: only {pool_n} held-out texts pass the filter, so N stops at {N_MAX}. "
          "The -1 slope predicts a single survivor near N=8400; if N_MAX is well below that, "
          "report the extrapolation as an extrapolation.")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CFG.model_id, cache_dir=CFG.hf_cache_dir, revision=CFG.model_revision)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    CFG.model_id, cache_dir=CFG.hf_cache_dir, revision=CFG.model_revision,
    torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, attn_implementation="sdpa").eval().to(DEVICE)
assert not bool(getattr(model.config, "tie_word_embeddings", True)), "Expected an untied head."
V, D = model.lm_head.weight.shape
W_ORIGINAL = model.lm_head.weight.detach().clone()      # kept; the audit copies and damages it

def hidden_states(items, name):
    path = CACHE / f"hidden_{name}_{len(items)}.pt"
    if path.exists():
        return torch.load(path, map_location="cpu", weights_only=True)
    out = []
    for s in tqdm(range(0, len(items), CFG.feature_batch), desc=f"features:{name}"):
        enc = tokenizer(items[s:s + CFG.feature_batch], return_tensors="pt", padding=True,
                        truncation=True, max_length=CFG.context_tokens).to(DEVICE)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            h = model.model(**enc, use_cache=False, return_dict=True).last_hidden_state
        out.append(h[:, -1].float().cpu())
    h = torch.cat(out); torch.save(h, path); return h

H_POOL = hidden_states(pool_texts, "pool")
H_CAL = hidden_states(cal_texts, "cal")
assert torch.equal(H_POOL, H_POOL.to(torch.bfloat16).float()), "Feature cache is not exactly BF16."
weight_rms = float(W_ORIGINAL.float().square().mean().sqrt())
rho = CFG.rho_multiple * weight_rms
tamper_step = CFG.tamper_fraction_of_bound * rho
row_scale = W_ORIGINAL.float().abs().amax(dim=1).clamp_min(1e-12) / 127.0
del model; gc.collect(); torch.cuda.empty_cache()
print({"vocab": V, "hidden": D, "rho": rho, "tamper_step": tamper_step})

In [ ]:
def qcode(z):
    if bool(((z < -R) | (z >= R)).any()):
        raise RuntimeError("Public logit range saturated; the record convention is violated.")
    return torch.floor((z + R) / WIDTH).to(torch.uint8)

def codes_for(h, head, name):
    """Retained codes, computed by the SAME path the screen later uses."""
    path = CACHE / f"codes_{name}_{len(h)}_y{CFG.output_bits}.pt"
    if path.exists():
        return torch.load(path, map_location="cpu", weights_only=True)
    out = []
    for s in tqdm(range(0, len(h), CFG.head_batch), desc=f"codes:{name}"):
        out.append(qcode(h[s:s + CFG.head_batch].to(DEVICE, torch.float32) @ head.T).cpu())
    x = torch.cat(out); torch.save(x, path); return x

CODES_POOL = codes_for(H_POOL, W_ORIGINAL.float(), "pool")

# Retained order: distinct inputs unless replacement is explicitly requested.
rng = np.random.default_rng(CFG.pilot_seed + 3)
order = (rng.choice(len(H_POOL), N_MAX, replace=True) if CFG.sample_with_replacement
         else rng.permutation(len(H_POOL))[:N_MAX])
retained_H = H_POOL[order].to(DEVICE, torch.float32)
retained_codes = CODES_POOL[torch.as_tensor(order)]        # kept on CPU; sliced per use
REPORT["pool"]["distinct_inputs"] = int(len(np.unique(order)))
print({"retained_H": tuple(retained_H.shape), "distinct_inputs": REPORT["pool"]["distinct_inputs"]})

In [ ]:
# Public calibration rule (salience pools) and their uniform counterparts.
with torch.inference_mode():
    hcal = H_CAL.to(DEVICE, torch.float32)
    logits = hcal @ W_ORIGINAL.float().T
    salient_rows = torch.topk(torch.softmax(logits, -1).mean(0), min(256, V)).indices.cpu().numpy()
    salient_cols = torch.topk(H_CAL.square().mean(0), min(512, D)).indices.cpu().numpy()
    feature_energy = H_CAL.square().mean(0).numpy()
    del logits, hcal; torch.cuda.empty_cache()

_r = np.random.default_rng(CFG.pilot_seed + 11)
uniform_rows = _r.choice(V, len(salient_rows), replace=False)
uniform_cols = _r.choice(D, len(salient_cols), replace=False)
POOLS = {
    "salient":                   (salient_rows, salient_cols),
    "uniform":                   (uniform_rows, uniform_cols),
    "salient_rows_uniform_cols": (salient_rows, uniform_cols),
    "uniform_rows_salient_cols": (uniform_rows, salient_cols),
}
MODE_INDEX = {m: i for i, m in enumerate(sorted(POOLS))}   # deterministic seeds
REPORT["pools"] = {
    "median_feature_energy_salient_cols": float(np.median(feature_energy[salient_cols])),
    "median_feature_energy_uniform_cols": float(np.median(feature_energy[uniform_cols])),
    "energy_ratio": float(np.median(feature_energy[salient_cols])
                          / max(np.median(feature_energy[uniform_cols]), 1e-30)),
}
print(json.dumps(REPORT["pools"], indent=2))
print("\nThis ratio is the mechanism claim: interval width goes like WIDTH/(N*|h_j|), so the "
      "candidate counts below should differ by roughly its square root per coordinate.")

## EVALUATOR ONLY — tamper factory

One changed weight in each of `sparsity` distinct rows, drawn from the named pool. INT8-derived,
rho-bounded, rounded to BF16 — the same contract as the frozen demo.

In [ ]:
def make_tamper(mode, W, trial=None):
    """EVALUATOR ONLY. Returns the coordinates and the damaged values."""
    rows_pool, cols_pool = POOLS[mode]
    rng = np.random.default_rng(CFG.pilot_seed + 10_000 * (trial or CFG.tamper_trial_id)
                                + 137 * MODE_INDEX[mode])
    rows = rng.choice(rows_pool, CFG.sparsity, replace=False).astype(np.int64)
    cols = rng.choice(cols_pool, CFG.sparsity, replace=True).astype(np.int64)
    signs = rng.choice([-1, 1], CFG.sparsity).astype(np.int64)
    rr = torch.as_tensor(rows, device=DEVICE); cc = torch.as_tensor(cols, device=DEVICE)
    scale = row_scale[rr]
    old_signed = torch.round(W[rr, cc].float() / scale).clamp(-127, 127).to(torch.int64)
    step = torch.round(torch.full_like(scale, tamper_step) / scale).clamp_min(1).to(torch.int64)
    step = torch.minimum(step, torch.floor(torch.full_like(scale, rho) / scale).to(torch.int64))
    sign = torch.as_tensor(signs, device=DEVICE)
    proposed = (old_signed + sign * step).clamp(-127, 127)
    proposed = torch.where(proposed == old_signed,
                           (old_signed - sign * step).clamp(-127, 127), proposed)
    if bool((proposed == old_signed).any()):
        raise RuntimeError(f"[{mode}] INT8 reference change vanished.")
    old = W[rr, cc].clone()
    new = (old.float() + (proposed - old_signed).float() * scale).to(torch.bfloat16)
    if bool((new == old).any()) or float((new.float() - old.float()).abs().max()) > rho:
        raise RuntimeError(f"[{mode}] BF16 tamper violates the frozen contract.")
    assert len(np.unique(rows)) == CFG.sparsity, "dispersed layout needs distinct rows"
    return {"mode": mode, "rows": rows, "cols": cols, "old": old, "new": new, "rr": rr, "cc": cc}

## Item 1 — calibrate the screen tolerance

Two measurements, then a choice.

**Replay discrepancy** recomputes the *original* logits at two batch shapes and reports how far
they move. No tamper, no cached codes: it isolates the FP32 accumulation effect. v1's 64 false
rows correspond to `eps ~ 5e-7`; the expected count is
$V\,(1-(1-2\eta/\text{WIDTH})^{N})$, so this predicts the false-positive count directly.

**The eta sweep** flags a row only when its recomputed logit leaves the retained cell by more than
`eta`. `eta = 0` is the exact code comparison v1 used. Pick the smallest `eta` with recall 1.0 and
no false positives, and declare it as part of the record convention.

In [ ]:
rep = ia.replay_discrepancy(retained_H[:256], W_ORIGINAL.float(), R, WIDTH, batches=(32, 8))
eps = rep["max_abs_logit_diff"]
REPORT["item1_replay"] = dict(rep)
REPORT["item1_replay"]["predicted_false_rows_at_eta0"] = [
    {"N": int(n), "expected": float(V * (1 - (1 - min(1.0, 2 * eps / WIDTH)) ** n))}
    for n in (256, 1024, N_MAX)]
print(json.dumps(REPORT["item1_replay"], indent=2))
print(f"\nA screen tolerance must exceed {eps:.3e}; the interval slack "
      f"({CFG.constraint_slack:.0e}) already does.")

In [ ]:
# EVALUATOR ONLY scoring of the sweep. Uses the salient tamper as the reference instance.
W = W_ORIGINAL.clone()
_t = make_tamper("salient", W)
with torch.no_grad():
    W[_t["rr"], _t["cc"]] = _t["new"]

sweep_rows, worst_excursion = ia.screen_eta_sweep(
    retained_H[:256], retained_codes[:256], W.float(), R, WIDTH, CFG.eta_grid,
    _t["rows"], chunk=CFG.head_batch)
S = pd.DataFrame(sweep_rows)
clean = S[(S["recall"] == 1.0) & (S["false_positives"] == 0)]
ETA = float(clean["eta"].min()) if len(clean) else float(S[S["recall"] == 1.0]["eta"].max())
REPORT["item1_eta_sweep"] = {"grid": sweep_rows, "chosen_eta": ETA,
                             "clean_eta_exists": bool(len(clean))}
print(S.to_string(index=False))
print(f"\nchosen eta = {ETA:g}")
if not len(clean):
    print("NOTE: no tolerance removes every false positive at full recall. Report the "
          "recall/false-positive trade-off rather than a single operating point.")
with torch.no_grad():
    W[_t["rr"], _t["cc"]] = _t["old"]
del W, _t; gc.collect(); torch.cuda.empty_cache()

## Items 2 and 3 — the audit across coordinate pools

For each tamper pool: screen at the calibrated `eta`, compute exact intervals per changed row,
count candidates exactly (or estimate when the set is too large), and price the ladder.

`uniform` is the control that decides whether the headline generalises. Expect lower row recall
there: a change on a low-energy column may not move any code at all.

In [ ]:
def audit_rows(W_dam, rows_to_audit, hN, codes_all, verify_cap, draws, rng, label=""):
    """Per-row candidate counts. Exact where affordable, estimated otherwise."""
    recs = []
    for row in tqdm(rows_to_audit, desc=f"intervals:{label}", leave=False):
        z = W_dam[int(row)].float()
        cr = codes_all[:, int(row)].to(DEVICE)
        lo, hi, feas = ia.row_intervals(z, hN, cr, R, WIDTH, CFG.constraint_slack, rho,
                                        chunk=CFG.interval_chunk)
        cnt, left, right = ia.count_candidates(z, lo, hi, feas, VALUES)
        exact, per_col = ia.verify_row(z, hN, cr, left, right, cnt, VALUES, R, WIDTH, verify_cap)
        if exact is None:
            est = ia.estimate_row(z, hN, cr, left, right, cnt, VALUES, R, WIDTH,
                                  n_samples=draws, rng=rng)
            recs.append({"row": int(row), "method": "estimate", "candidates": est["estimate"],
                         "lower": est["lower"], "upper": est["upper"],
                         "feasible_columns": int((cnt > 0).sum()),
                         "interval_upper": int(cnt.sum())})
        else:
            recs.append({"row": int(row), "method": "exact", "candidates": float(exact),
                         "lower": float(exact), "upper": float(exact),
                         "feasible_columns": int((per_col > 0).sum()),
                         "interval_upper": int(cnt.sum())})
        recs[-1]["_counts"] = cnt if exact is None else per_col
    return recs


def ladder(n_flagged, residual_bits, s=CFG.sparsity):
    base = 2 * s * 31
    rows = [("data-free full-head RS (2s checks, p=2^31-1)", base)]
    if s == 64:                            # the v1 number is specific to that frozen instance
        rows.append(("as reported in v1: l1 top-16 flags + RS", 114 * 31))
    rows += [("row-summary code + screen (2|E| checks, p=2^17-1)", 2 * n_flagged * 17),
            ("row locators + interval column ID (|E| checks, p=2^17-1)", n_flagged * 17),
            ("information left by the record (sum log2 candidates)", int(math.ceil(residual_bits)))]
    return pd.DataFrame([{"scheme": k, "bits": v, "bytes": math.ceil(v / 8),
                          "vs_data_free": f"{100 * (1 - v / base):.1f}%"} for k, v in rows])

In [ ]:
N_REF = 256 if 256 in N_GRID else N_MAX          # keep v1 comparability
hN = retained_H[:N_REF]
codes_N = retained_codes[:N_REF]
mode_results, mode_tables = {}, {}

for mode in CFG.pool_modes:
    W = W_ORIGINAL.clone()
    t = make_tamper(mode, W)
    with torch.no_grad():
        W[t["rr"], t["cc"]] = t["new"]
    truth = set(int(r) for r in t["rows"])

    flagged, _ = ia.screen_rows(hN, codes_N, W.float(), R, WIDTH, ETA, chunk=CFG.head_batch)
    caught = truth & set(int(r) for r in flagged)
    recall = len(caught) / len(truth)

    recs = audit_rows(W, np.array(sorted(truth)), hN, codes_N,
                      CFG.verify_cap, CFG.estimate_draws,
                      np.random.default_rng(CFG.pilot_seed + 5), label=mode)
    # EVALUATOR ONLY: does the true coordinate survive?
    for r in recs:
        k = int(np.flatnonzero(t["rows"] == r["row"])[0])
        r["true_column_survives"] = bool(r.pop("_counts")[int(t["cols"][k])] > 0)
    df = pd.DataFrame(recs); df["mode"] = mode
    residual = float(np.log2(np.maximum(df["candidates"].values, 1)).sum())

    mode_results[mode] = {
        "flagged_rows": int(len(flagged)), "row_recall": recall,
        "false_positive_rows": int(len(flagged) - len(caught)),
        "rows_with_column_pinned": int((df["feasible_columns"] == 1).sum()),
        "rows_fully_pinned": int((df["candidates"] <= 1).sum()),
        "median_candidates": float(df["candidates"].median()),
        "median_bits_per_row": float(np.log2(max(df["candidates"].median(), 1))),
        "residual_bits": residual,
        "rows_exact": int((df["method"] == "exact").sum()),
        "true_column_survives_all": bool(df["true_column_survives"].all()),
        "screen_usable": recall == 1.0,
    }
    mode_tables[mode] = df
    with torch.no_grad():
        W[t["rr"], t["cc"]] = t["old"]
    print(f"{mode:<28} recall {recall:>5.2f}  flagged {len(flagged):>4}  "
          f"median {df['candidates'].median():>12,.0f}  residual {residual:>8.0f} bits")

MODES = pd.DataFrame(mode_results).T
MODES.to_csv(OUT / "by_pool_mode.csv")
pd.concat(mode_tables.values()).to_csv(OUT / "per_row_by_mode.csv", index=False)
REPORT["item3_pool_modes"] = mode_results
print()
print(MODES.to_string())

In [ ]:
print("Ladder per pool mode (bits):\n")
for mode, res in mode_results.items():
    L = ladder(res["flagged_rows"], res["residual_bits"])
    print(f"--- {mode}  (screen usable: {res['screen_usable']}) ---")
    print(L.to_string(index=False)); print()
    REPORT.setdefault("item3_ladders", {})[mode] = L.to_dict(orient="records")
    L.to_csv(OUT / f"ladder_{mode}.csv", index=False)

## Item 4 — the extended N sweep

Same measurement across the enlarged grid, now recording `feasible_columns` so the crossover
between the location regime and the value regime is a recorded fact rather than an inference.
Points too large to enumerate are estimated, so every point is a count rather than a bound.

In [ ]:
sweep = []
for mode in CFG.sweep_modes:
    W = W_ORIGINAL.clone()
    t = make_tamper(mode, W)
    with torch.no_grad():
        W[t["rr"], t["cc"]] = t["new"]
    rows_m = np.array(sorted(set(int(r) for r in t["rows"])))
    for n in tqdm(N_GRID, desc=f"N sweep:{mode}"):
        recs = audit_rows(W, rows_m, retained_H[:n], retained_codes[:n],
                          CFG.verify_cap, CFG.estimate_draws,
                          np.random.default_rng(CFG.pilot_seed + n), label=f"{mode}/N={n}")
        for r in recs:
            r.pop("_counts", None)
            sweep.append({"mode": mode, "N": n, **r})
    with torch.no_grad():
        W[t["rr"], t["cc"]] = t["old"]

SW = pd.DataFrame(sweep); SW.to_csv(OUT / "sweep_vs_N.csv", index=False)

REPORT["item4_law"] = {}
for mode in CFG.sweep_modes:
    sub = SW[SW["mode"] == mode]
    med = sub.groupby("N")["candidates"].median().clip(lower=1)
    cols = sub.groupby("N")["feasible_columns"].median()
    # The value regime starts once the column is pinned; fit the slope only there.
    pinned = cols[cols <= 1]
    n_star = int(pinned.index.min()) if len(pinned) else None
    live = med[(med.index >= (n_star or 10 ** 9)) & (med > 1)]
    if len(live) >= 2:                     # 2 points give a secant, 3+ a least-squares fit
        slope = float(np.polyfit(np.log2(live.index.values.astype(float)),
                                 np.log2(live.values), 1)[0])
    else:
        slope = float("nan")               # column pinned only at the last N: extend the grid
    REPORT["item4_law"][mode] = {
        "median_candidates_by_N": {int(k): float(v) for k, v in med.items()},
        "median_feasible_columns_by_N": {int(k): float(v) for k, v in cols.items()},
        "N_column_pinned": n_star,
        "value_regime_slope": slope,
        "n_fit_points": int(len(live)),
        "predicted_slope": -1.0,
        "N_median_reaches_1": int(med[med <= 1].index.min()) if (med <= 1).any() else None,
        "extrapolated_N_for_1": (None if not len(live) else
                                 float(live.index[-1] * live.values[-1])),
    }
print(json.dumps(REPORT["item4_law"], indent=2))

In [ ]:
# Two panels sharing one y-axis: the column count (location regime) and the candidate
# count (value regime). Rows are a distribution, so the median carries the reading.
INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e2e2e2"
HUE = {"salient": "#2f6f9f", "uniform": "#b4622a"}
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.3), dpi=140)
for mode in CFG.sweep_modes:
    sub = SW[SW["mode"] == mode]
    med = sub.groupby("N")["candidates"].median().clip(lower=1)
    cols = sub.groupby("N")["feasible_columns"].median().clip(lower=1)
    axes[0].plot(cols.index, cols.values, color=HUE[mode], lw=2, marker="o", ms=5, label=mode)
    axes[1].plot(med.index, med.values, color=HUE[mode], lw=2, marker="o", ms=5, label=mode)
ref_n = np.array(N_GRID, dtype=float)
anchor = SW[SW["mode"] == CFG.sweep_modes[0]].groupby("N")["candidates"].median().iloc[-1]
axes[1].plot(ref_n, np.maximum(anchor * (ref_n / ref_n[-1]) ** -1.0, 1), color=INK, lw=1.5,
             ls="--", label="slope $-1$")
axes[0].set_ylabel("median feasible columns per changed row")
axes[1].set_ylabel("median surviving weight values per changed row")
axes[0].set_title("location regime", loc="left"); axes[1].set_title("value regime", loc="left")
for ax in axes:
    ax.set_xscale("log", base=2); ax.set_yscale("log", base=2)
    ax.set_xlabel("retained predictions $N$")
    ax.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(MUTED)
    ax.tick_params(colors=MUTED); ax.xaxis.label.set_color(INK); ax.yaxis.label.set_color(INK)
    ax.legend(frameon=False, labelcolor=INK)
fig.tight_layout(); fig.savefig(OUT / "two_regimes.png", bbox_inches="tight"); plt.show()

## Item 2 — head profile, estimated rather than capped

The same measurement on untouched rows, with subsample verification so every number is a count
with a confidence interval instead of an interval upper bound. These candidates share the
retained transcript **and** a common damaged checkpoint, so `s` times the per-row log2 lower-bounds
every encoder for an `s`-sparse tamper on comparable rows. Quote the `lower` column for a converse.

In [ ]:
W = W_ORIGINAL.clone()                              # untouched rows: damaged == original
rng = np.random.default_rng(CFG.pilot_seed + 7)
sample_rows = rng.choice(V, min(CFG.head_profile_rows, V), replace=False)
recs = audit_rows(W, sample_rows, hN, codes_N, CFG.verify_cap, CFG.estimate_draws,
                  rng, label="head profile")
for r in recs: r.pop("_counts", None)
PF = pd.DataFrame(recs); PF["energy"] = np.nan
PF.to_csv(OUT / "head_profile.csv", index=False)

lo_bits = np.log2(np.maximum(PF["lower"].values, 1))
REPORT["item2_head_profile"] = {
    "rows_sampled": len(PF),
    "fraction_exact": float((PF["method"] == "exact").mean()),
    "median_candidates": float(PF["candidates"].median()),
    "median_bits": float(np.log2(max(PF["candidates"].median(), 1))),
    "median_bits_lower_bound": float(np.median(lo_bits)),
    "p10_bits": float(np.quantile(np.log2(np.maximum(PF["candidates"].values, 1)), .10)),
    "p90_bits": float(np.quantile(np.log2(np.maximum(PF["candidates"].values, 1)), .90)),
    "converse_bits_for_s_rows": float(CFG.sparsity * np.median(lo_bits)),
    "note": "converse uses the one-sided lower confidence bound on each row's count",
}
print(json.dumps(REPORT["item2_head_profile"], indent=2))
print(f"\nlower-bound on any encoder for s={CFG.sparsity} comparable rows: "
      f"{REPORT['item2_head_profile']['converse_bits_for_s_rows']:.0f} bits "
      f"(data-free full-head RS baseline is {2 * CFG.sparsity * 31} bits)")

In [ ]:
REPORT["config"] = {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.__dict__.items()}
REPORT["provenance"] = {"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
                        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S")}
(OUT / "audit_v2_report.json").write_text(json.dumps(REPORT, indent=2, default=float))
print("wrote", OUT / "audit_v2_report.json")
print(json.dumps({k: REPORT[k] for k in REPORT if k.startswith("item")}, indent=2, default=float)[:3500])

## What the numbers mean

**Item 1.** If the chosen `eta` gives recall 1.0 with zero false positives, `|E|` drops to 64 and
every construction halves: the row-locator rung goes 2,176 -> 1,088 bits (72.6% below data-free),
and the row-summary scheme stops being worse than doing nothing. If no `eta` is clean, the screen
has a genuine precision limit and the ladder must be priced at the observed `|E|` — say so rather
than quoting the optimistic rung.

**Item 2.** `converse_bits_for_s_rows` is the paper's lower bound. If it lands near the 3,968-bit
data-free baseline on typical coordinates, the honest claim is that the record helps a lot on
salient weights and little elsewhere. If it lands well below, the record is broadly informative
and the headline generalises.

**Item 3.** The four pool modes separate the mechanism. If `salient_rows_uniform_cols` behaves
like `uniform` and `uniform_rows_salient_cols` behaves like `salient`, the informative variable is
**column feature energy**, which is what `width(I) ~ WIDTH/(N|h_j|)` predicts. That turns the
mechanism from an assertion into a controlled result. Watch for `uniform` row recall below 1.0:
a change on a low-energy column may move no code at all, which is itself the cleanest statement
of what the record cannot see.

**Item 4.** The value-regime slope is now fitted only after the column is pinned, so it is a clean
estimate rather than a mixture. Confirming `-1` out to large `N`, and observing the median reach 1,
turns the extrapolated `N ~ 8,400` into a measurement. If the curve flattens early instead, the
pool has saturated — check `distinct_inputs` before concluding anything about a floor.

**Then stop.** These four close the gaps in the existing story. They do not open a new one.